In [0]:
%scala
//Load latest snapshot

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.apache.sedona.spark.SedonaContext
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo

import org.apache.spark.sql.{Dataset, SparkSession}
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.{LoadFreshSnapshotData, LoadDataFromMcr}
import com.databricks.dbutils_v1.DBUtilsHolder
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader

  import com.tomtom.addressing.bulk.commons.config.ConfigLoader


val mapper = new ObjectMapper()
    mapper.configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
      .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
      .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD" //dbutils.widgets.get("environment").toUpperCase()
println(s"Environment Name $env")

implicit val sparky = spark
// implicit val envConfig = ConfigLoader.forEnvironment(env.toLowerCase)

ConfigLoader.forEnvironment(env.toLowerCase)   

println(ConfigLoader.getConfig.getString("secrets.key-vault.url"))


val latestRevision = LoadSnapshotInfo.getLatestRevisionId(14533)
print("latest revision: ", latestRevision)
// val database = "data_recovery"
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(14533, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

val config = SourceConfigLoader.getConfig

val deltaConfig = config.getTableMapping.getOrDefault(env,"preprocess_dev.source_data")
val Array(databaseName, tableName) = deltaConfig.split("\\.")
println(s"Database Name: $databaseName")
println(s"Table Name: $tableName")


SparkHelper.init("data_recovery")


new LoadFreshSnapshotData().run()

// changelet
// 42429707
// snapshot
// 42037615

In [0]:
%scala

import org.apache.spark.sql.functions._
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository


val aptDS = new OrbisElementRepository("14533").readAll

display(aptDS)

In [0]:
%scala
val countryTagKey = "metadata:country"

val country_wise_list = aptDS
  // Filter: metadata:apa:improvement = yes
  .filter(exists(col("tags"), t => 
    t.getField("tagKey").getField("key") === "metadata:apa:improvement" 
    && t.getField("value") === "yes"
  ))
  // Extract country value from tags array
  .withColumn("country", 
    filter(col("tags"), t => 
      t.getField("tagKey").getField("key") === countryTagKey
    ).getItem(0).getField("value")
  )

val country_wise_count = country_wise_list
  .groupBy("country")
  .count()
  .orderBy(desc("count"))

display(country_wise_count)

// USA	30853221
// ESP	5598678 -- 11-may
// ESP  5545261 -- 28-May

In [0]:
%scala
import org.apache.spark.sql.functions.col
import com.tomtom.orbis.io.spark.model.Id
import java.lang.Long

def convertOrbisIdToString(orbisId: Id): String = {
  val COLON_SEPARATOR = ":"
  Seq(orbisId.layerId.getOrElse(14533).toString, 
      Long.toUnsignedString(orbisId.high), 
      Long.toUnsignedString(orbisId.low)).mkString(COLON_SEPARATOR)
}

// UDF to apply the function to the DataFrame
val convertOrbisIdUDF = udf((orbisId: Id) => convertOrbisIdToString(orbisId))

val countryTagKey = "metadata:country"

val relocated_apt = aptDS
  // Filter: metadata:apa:improvement = yes
  .filter(exists(col("tags"), t => 
    t.getField("tagKey").getField("key") === "metadata:apa:improvement" 
    && t.getField("value") === "yes"
  ))
  // Extract country value from tags array
  .withColumn("country", 
    filter(col("tags"), t => 
      t.getField("tagKey").getField("key") === countryTagKey && t.getField("value") === "ESP"
    ).getItem(0).getField("value")
  )
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))


display(relocated_apt)

In [0]:
%scala
val countryTagKey = "metadata:country" 
val stateTagKey   = "addr:state:en-Latn"

val state_wise_list = aptDS
    // Filter: country = USA
    .filter(exists(col("tags"), t => t.getField("tagKey").getField("key") ===  countryTagKey 
    && t.getField("value") === "USA"
    ))
     // Filter: metadata:apa:improvement = yes
    .filter(exists(col("tags"), t =>  t.getField("tagKey").getField("key") === "metadata:apa:improvement" 
    && t.getField("value") ===  "yes"))            
    // Extract state value from tags array
    .withColumn ("state", filter(col("tags"), t => t.getField("tagKey").getField("key") ===  stateTagKey).getItem(0).getField("value"))

val state_wise_count = state_wise_list.groupBy("state")
    .count()
    .orderBy(desc("state"))

display(state_wise_count)

In [0]:
%scala

val filtered_state = state_wise_list.filter(col("state") === "QC")
display(filtered_state)